In [20]:
from pypdf import PdfReader
import os

from pathlib import Path
import re
import pypdf

from dotenv import load_dotenv
load_dotenv('.env')
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document



In [21]:
def read_pdf(filepath: str) -> str:
    """Reads a PDF and returns the full extracted text."""
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# Usage
pdf_path = os.path.join("data", "Shravan Resume Rag feed.pdf")
content = read_pdf(pdf_path)
print(content)


                    
 
 
SHRAVAN A. KULKARNI 
 
 
 
 
Senior Engineer with 12+ years of experience delivering high-impact solutions for Fortune 500 companies, 
Industry leaders, across global domains. Expertise in frontend architecture and UI engineering, currently 
transitioning into Agentic AI Engineering—focusing on autonomous agent workflows, LLM integration, RAG 
Pipelines and human-in-the-loop interfaces. Seeking to leverage deep software design experience to build next-
generation AI products with a forward-thinking engineering team. 
 
 
 
 
 
● 10 years of work experience in WEB DEVELOPMENT, creating pixel perfect screens, complicated 
workflows with Responsive UI and security features. 
● Experience on up to date advance web development technologies Angular, Node JS, ,Java Script, Type Script, 
CSS3, HTML5 with supporting libraries like NgRx and RxJS . 
● Skilled in Agentic workflows like LLM integration( with all major models like OpenAI, DeepSeek, Gemini..) 
RAG pipelines,

In [22]:
from pathlib import Path
import re
import pypdf

# 1. Path to your PDF file
pdf_path = Path("data/Shravan Resume Rag feed.pdf")

# 2. Extract full text from all pages
reader = pypdf.PdfReader(pdf_path)
full_text = "\n".join([page.extract_text() or "" for page in reader.pages])

# 3. Two-level chunking:
#    Step A: Split sections separated by > 3 newlines (>= 4 newlines)
section_pattern = r"(?:\r?\n[ \t]*){4,}"
sections = [s.strip() for s in re.split(section_pattern, full_text) if s.strip()]

#    Step B: Split each section by bullet characters (●, •, \uf0b7, , -, *)
bullet_regex = r"(?:^|\n)[ \t]*[●•\uf0b7\-\*][ \t]*"

chunks = []
for sec in sections:
    bullet_items = [b.strip() for b in re.split(bullet_regex, sec) if b.strip()]
    if len(bullet_items) > 1:
        # Section has bullet points -> each bullet becomes an individual chunk
        chunks.extend(bullet_items)
    elif bullet_items:
        # No bullets in this section -> keep as single chunk
        chunks.append(bullet_items[0])

# 4. View results
print(f"Total chunks created: {len(chunks)}\n")

for idx, chunk in enumerate(chunks, start=1):
    print("=" * 60)
    print(f"CHUNK {idx} (Length: {len(chunk)} characters)")
    print("=" * 60)
    print(chunk)
    print("\n")


Total chunks created: 25

CHUNK 1 (Length: 19 characters)
SHRAVAN A. KULKARNI


CHUNK 2 (Length: 492 characters)
Senior Engineer with 12+ years of experience delivering high-impact solutions for Fortune 500 companies, 
Industry leaders, across global domains. Expertise in frontend architecture and UI engineering, currently 
transitioning into Agentic AI Engineering—focusing on autonomous agent workflows, LLM integration, RAG 
Pipelines and human-in-the-loop interfaces. Seeking to leverage deep software design experience to build next-
generation AI products with a forward-thinking engineering team.


CHUNK 3 (Length: 144 characters)
10 years of work experience in WEB DEVELOPMENT, creating pixel perfect screens, complicated 
workflows with Responsive UI and security features.


CHUNK 4 (Length: 166 characters)
Experience on up to date advance web development technologies Angular, Node JS, ,Java Script, Type Script, 
CSS3, HTML5 with supporting libraries like NgRx and RxJS .


CHUNK 5 (L

In [ ]:


# 1. Convert chunks into LangChain Document objects with metadata
documents = [
    Document(
        page_content=chunk,
        metadata={"source": pdf_path.name, "chunk_id": idx}
    )
    for idx, chunk in enumerate(chunks, start=1)
]

# 2. Initialize the OpenAI embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Create FAISS vector store from the documents
print("Embedding chunks and creating FAISS index...")
vector_store = FAISS.from_documents(documents, embeddings)
print("FAISS vector store created successfully!")

# 4. Save the index locally to disk
index_dir = "faiss_index"
vector_store.save_local(index_dir)
print(f"Vector store saved to '{index_dir}'")

# 5. Test similarity search retrieval
query = "What are Shravan's education qualification"
results = vector_store.similarity_search(query, k=4)

print(f"\n--- Top matches for: '{query}' ---")
for doc in results:
    print(f"\n[Chunk {doc.metadata['chunk_id']}] ({doc.metadata['source']}):")
    print(doc.page_content)
    print("-" * 60)
